# Phase 9: Power BI Dashboard Data Preparation

**Project:** Customer Churn Prediction & Retention Analytics  
**Source Input:** `outputs/predictions/customer_retention_recommendations.csv`  
**Target Directory:** `powerbi/`  

### Objectives:
1. Ingest the prioritized customer retention dataset ($N = 7,043$).
2. Engineer presentation-ready analytical dimensions for Power BI:
   - **Tenure Group:** Standardized lifecycle cohorts (`0-12 Months`, `13-24 Months`, etc.).
   - **Monthly Charge Band:** Data-driven quartile segments (`Low Charges`, `Lower-Mid Charges`, `Upper-Mid Charges`, `High Charges`).
   - **Churn Status & Predicted Churn Status:** User-friendly text descriptors for slicers.
   - **Risk Sort Order:** Numerical index (`1`, `2`, `3`) to ensure correct visualization sorting in Power BI.
3. Perform complete data integrity validation.
4. Export `powerbi/customer_churn_dashboard_data.csv`, `powerbi/customer_churn_kpi_summary.csv`, and `powerbi/high_risk_customers.csv`.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

out_dir = Path("../powerbi")
out_dir.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

## 2. Load Prioritized Retention Dataset

In [ ]:
in_path = Path("../outputs/predictions/customer_retention_recommendations.csv")
df = pd.read_csv(in_path)
print(f"Loaded {len(df):,} customer records with {len(df.columns)} columns.")

## 3. Compute Monthly Charges Quartiles

In [ ]:
q25 = float(df["Monthly Charges"].quantile(0.25))
q50 = float(df["Monthly Charges"].quantile(0.50))
q75 = float(df["Monthly Charges"].quantile(0.75))
min_c = float(df["Monthly Charges"].min())
max_c = float(df["Monthly Charges"].max())

print(f"Data-Driven Quartile Cutoffs for Monthly Charges:")
print(f"- Minimum:                       ${min_c:.2f}")
print(f"- 25th Percentile (Q1):          ${q25:.2f}")
print(f"- 50th Percentile / Median (Q2): ${q50:.2f}")
print(f"- 75th Percentile (Q3):          ${q75:.2f}")
print(f"- Maximum:                       ${max_c:.2f}")

## 4. Engineer Analytical Dimension Columns

In [ ]:
# 1. Tenure Group
def tenure_group(months):
    if months <= 12: return "0-12 Months"
    elif months <= 24: return "13-24 Months"
    elif months <= 36: return "25-36 Months"
    elif months <= 48: return "37-48 Months"
    elif months <= 60: return "49-60 Months"
    else: return "61+ Months"

df["Tenure Group"] = df["Tenure Months"].apply(tenure_group)

# 2. Monthly Charge Band
def charge_band(c):
    if c <= q25: return f"Low Charges (<=${q25:.2f})"
    elif c <= q50: return f"Lower-Mid Charges (${q25:.2f}-${q50:.2f})"
    elif c <= q75: return f"Upper-Mid Charges (${q50:.2f}-${q75:.2f})"
    else: return f"High Charges (>${q75:.2f})"

df["Monthly Charge Band"] = df["Monthly Charges"].apply(charge_band)

# 3. Status Labels for Filters
df["Churn Status"] = df["Churn Value"].apply(lambda v: "Churned" if v == 1 else "Retained")
df["Predicted Churn Status"] = df["Predicted Churn"].apply(lambda v: "Predicted Churn" if v == 1 else "Predicted Retained")

# 4. Risk Sorting Order (for Power BI sort-by-column functionality)
risk_sort_map = {"Low Risk": 1, "Medium Risk": 2, "High Risk": 3}
df["Risk Sort Order"] = df["Risk Level"].map(risk_sort_map)

print("Dimensions engineered successfully.")

## 5. Validate Data Integrity

In [ ]:
assert len(df) == 7043, "Row count mismatch!"
assert df["CustomerID"].nunique() == 7043, "Duplicate Customer IDs detected!"
assert df["Churn Probability"].isna().sum() == 0, "Null churn probabilities!"
assert df["Risk Level"].isna().sum() == 0, "Null risk levels!"
assert df["Priority Rank"].isna().sum() == 0, "Null priority ranks!"
print("Integrity verification passed: 7,043 unique, clean records.")

## 6. Export Power BI Datasets

In [ ]:
# A. Main Dashboard Dataset
main_path = out_dir / "customer_churn_dashboard_data.csv"
df.to_csv(main_path, index=False)
print(f"Exported main dataset to {main_path.resolve()}")

# B. KPI Summary Table
kpi_df = pd.DataFrame([{
    "Total Customers": len(df),
    "Total Churned": int((df["Churn Value"] == 1).sum()),
    "Overall Churn Rate": round((df["Churn Value"] == 1).mean() * 100, 2),
    "High Risk Customers": int((df["Risk Level"] == "High Risk").sum()),
    "Medium Risk Customers": int((df["Risk Level"] == "Medium Risk").sum()),
    "Low Risk Customers": int((df["Risk Level"] == "Low Risk").sum()),
    "Average Monthly Charges": round(df["Monthly Charges"].mean(), 2),
    "Average Tenure": round(df["Tenure Months"].mean(), 2),
    "Average Churn Probability": round(df["Churn Probability"].mean() * 100, 2)
}])
kpi_path = out_dir / "customer_churn_kpi_summary.csv"
kpi_df.to_csv(kpi_path, index=False)
print(f"Exported KPI summary to {kpi_path.resolve()}")
display(kpi_df)

# C. High-Risk Action Table
high_risk_cols = [
    "Priority Rank", "CustomerID", "Churn Probability", "Contract", "Tenure Months",
    "Internet Service", "Payment Method", "Monthly Charges", "Tech Support",
    "Online Security", "Risk Drivers", "Retention Action"
]
high_risk_df = df[df["Risk Level"] == "High Risk"][high_risk_cols].sort_values(
    by="Priority Rank", ascending=True
).reset_index(drop=True)

hr_path = out_dir / "high_risk_customers.csv"
high_risk_df.to_csv(hr_path, index=False)
print(f"Exported High-Risk action table to {hr_path.resolve()}")
display(high_risk_df.head(10))